# Gene copy number feature extraction

We adopted the differences in gene expression between genes within and outside specific pathways to characterize the gene expression features of cell lines. The Mann–Whitney U test is a non-parametric method used to determine whether two independent samples originate from the same continuous distribution. Its fundamental principle involves comparing the ranks of observations between the two samples to assess whether the locations of the population distributions differ [75]. Consequently, we employed this test to compute the differences in gene expression between genes within and outside gene sets. We utilized the c2_kegg_medicus canonical pathway collection comprising 619 gene sets, ultimately constructing 619 pathway difference features for each cell line sample. The specific construction method is as follows:

In [ ]:
import pandas as pd
from scipy import stats

In [2]:
def remove_brackets(x):
    return x.split('(')[0].strip()

In [3]:
df_exp = pd.read_csv('/kaggle/input/ccle-geneexp/OmicsExpressionProteinCodingGenesTPMLogp1-23Q2.csv')
df_exp.columns = df_exp.columns.map(remove_brackets)
df_exp.head(5)

,Unnamed: 0,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,...,H3C2,H3C3,AC098582.1,DUS4L-BCAP29,C8orf44-SGK3,ELOA3B,NPBWR1,ELOA3D,ELOA3,CDR1
0,ACH-001113,4.331992,0.000000,7.364660,2.792855,4.471187,0.028569,1.226509,3.044394,6.500005,...,2.689299,0.189034,0.201634,2.130931,0.555816,0.0,0.275007,0.0,0.0,0.000000
1,ACH-001289,4.567424,0.584963,7.106641,2.543496,3.504620,0.000000,0.189034,3.813525,4.221877,...,1.286881,1.049631,0.321928,1.464668,0.632268,0.0,0.014355,0.0,0.0,0.000000
2,ACH-001339,3.150560,0.000000,7.379118,2.333424,4.228049,0.056584,1.310340,6.687201,3.682573,...,0.594549,1.097611,0.831877,2.946731,0.475085,0.0,0.084064,0.0,0.0,0.042644
3,ACH-001538,5.085340,0.000000,7.154211,2.545968,3.084064,0.000000,5.868390,6.165309,4.489928,...,0.214125,0.632268,0.298658,1.641546,0.443607,0.0,0.028569,0.0,0.0,0.000000
4,ACH-000242,6.729417,0.000000,6.537917,2.456806,3.867896,0.799087,7.208478,5.570159,7.127117,...,1.117695,2.358959,0.084064,1.910733,0.000000,0.0,0.464668,0.0,0.0,0.000000


In [4]:
df_geneSet = pd.read_csv('/kaggle/input/c2-cp-geneset/c2.cp.kegg_legacy.v2023.2.Hs.symbols.csv', sep='\t',header=None)
df_geneSet.head(5)

,0
0,"KEGG_ABC_TRANSPORTERS,ABCA1,ABCA10,ABCA12,ABCA..."
1,"KEGG_ACUTE_MYELOID_LEUKEMIA,AKT1,AKT2,AKT3,ARA..."
2,"KEGG_ADHERENS_JUNCTION,ACP1,ACTB,ACTG1,ACTN1,A..."
3,"KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,ACACB,ACS..."
4,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLIS...


In [5]:
# Retrieve all genes.
CCLEgeneSets = set(df_exp.columns.values[1:])
# Retrieve the name of the cell line.
cell_line_names = df_exp.iloc[0:,0]

In [6]:
U_test_result = []
pathways = []
# temp_ANOVA_counts = []
count = 0
for data in df_geneSet.values:
    geneSet_pathway = data[0].split(',')
    pathway,genes = geneSet_pathway[0],set(geneSet_pathway[1:])
    print(count+1,pathway)
    internal_genes = CCLEgeneSets & genes
    external_genes = CCLEgeneSets - internal_genes

    # Summarize the pathway data.
    pathways.append(pathway)

    # Internal data of the gene set.
    internal = df_exp[list(internal_genes)]
    # External data of the gene set.
    external = df_exp[list(external_genes)]

    
    U_stats = []
    
    for inter_data,exter_data in zip(internal.values,external.values):

        # Mann-Whitney U Test
        U_stat, p_value = stats.mannwhitneyu(inter_data.flatten(), exter_data.flatten(), alternative='two-sided')

        U_stats.append(p_value)

        # Check if there is a significant difference in variance.
        # F,p = stats.f_oneway(inter_data.flatten(), exter_data.flatten())
        # if p < 0.05:
        #     count+=1
    # print(count)
    count+=1
    U_test_result.append(U_stats)

1 KEGG_ABC_TRANSPORTERS
2 KEGG_ACUTE_MYELOID_LEUKEMIA
3 KEGG_ADHERENS_JUNCTION
4 KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY
5 KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM
6 KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION
7 KEGG_ALLOGRAFT_REJECTION
8 KEGG_ALPHA_LINOLENIC_ACID_METABOLISM
9 KEGG_ALZHEIMERS_DISEASE
10 KEGG_AMINOACYL_TRNA_BIOSYNTHESIS
11 KEGG_AMINO_SUGAR_AND_NUCLEOTIDE_SUGAR_METABOLISM
12 KEGG_AMYOTROPHIC_LATERAL_SCLEROSIS_ALS
13 KEGG_ANTIGEN_PROCESSING_AND_PRESENTATION
14 KEGG_APOPTOSIS
15 KEGG_ARACHIDONIC_ACID_METABOLISM
16 KEGG_ARGININE_AND_PROLINE_METABOLISM
17 KEGG_ARRHYTHMOGENIC_RIGHT_VENTRICULAR_CARDIOMYOPATHY_ARVC
18 KEGG_ASCORBATE_AND_ALDARATE_METABOLISM
19 KEGG_ASTHMA
20 KEGG_AUTOIMMUNE_THYROID_DISEASE
21 KEGG_AXON_GUIDANCE
22 KEGG_BASAL_CELL_CARCINOMA
23 KEGG_BASAL_TRANSCRIPTION_FACTORS
24 KEGG_BASE_EXCISION_REPAIR
25 KEGG_BETA_ALANINE_METABOLISM
26 KEGG_BIOSYNTHESIS_OF_UNSATURATED_FATTY_ACIDS
27 KEGG_BLADDER_CANCER
28 KEGG_BUTANOATE_METABOLISM
29 KEGG_B_CELL_RECEPTO

In [7]:
# row: pathway; column: cell line name.
GeneExp_T_test_Analysis = pd.DataFrame(U_test_result,index=pathways,
                                                   columns=cell_line_names)

In [8]:
GeneExp_T_test_Analysis.to_csv("GeneExp_Wilcoxon_test_Analysis_U_stats_C2_KEGG186.csv")